Audit the synthetic announcement log. `event_state_asof` hides rows that were not yet public and drops a name whose latest public action is `NONE`. The raw log still contains the earlier ADD.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from rebalance.calendar import TradingCalendar
from rebalance.config import load_research_config, repo_root, resolve_config_path
from rebalance.data.prices import load_fundamentals, load_prices
from rebalance.data.russell import load_announcements
from rebalance.data.securities import load_security_master
from rebalance.events.revisions import event_state_asof
from rebalance.events.timeline import select_tradable_events

root = repo_root()
config = load_research_config(root / "config" / "research.yaml")
calendar = TradingCalendar(config.calendar_name, config.timezone)
master = load_security_master(resolve_config_path(root, config.paths.security_master))
events = load_announcements(
    resolve_config_path(root, config.paths.announcements),
    master,
    timezone=config.timezone,
)
trades, excluded = select_tradable_events(events, calendar)
print("Synthetic fixtures only. Do not read these prints as a Russell result.")
print(f"announcements={len(events)} tradable={len(trades)} excluded={len(excluded)}")


Synthetic fixtures only. Do not read these prints as a Russell result.
announcements=23 tradable=20 excluded=1


In [2]:
x01 = events.loc[events["security_id"] == "X01", ["published_at", "action", "version"]]
print("X01 revision log")
display(x01)

may26 = event_state_asof(events, pd.Timestamp("2024-05-26 12:00", tz="America/New_York"))
print("X01 actions public on 26 May:", may26.loc[may26["security_id"] == "X01", "action"].tolist())

decision = calendar.session_open("2024-05-28") - pd.Timedelta(nanoseconds=1)
at_entry = event_state_asof(events, decision)
print("X01 actions just before the 28 May open:", at_entry.loc[at_entry["security_id"] == "X01", "action"].tolist())
print("ADD rows still stored in the log:", int((events["action"] == "ADD").sum()))
print(excluded.loc[excluded["security_id"] == "X01", ["security_id", "reason"]].to_string(index=False))
print("2024 entry sessions:", sorted(trades.loc[trades["rebalance_id"] == "2024-06", "entry_session"].astype(str).unique()))


X01 revision log


,published_at,action,version
21,2024-05-24 21:00:00+00:00,ADD,1
22,2024-05-27 22:00:00+00:00,NONE,2


X01 actions public on 26 May: ['ADD']
X01 actions just before the 28 May open: []
ADD rows still stored in the log: 12
security_id                 reason
        X01 cancelled_before_entry
2024 entry sessions: ['2024-05-28']
